In [5]:
import torch
from torch import nn
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [ ]:
# define model
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 24, 5, 2), nn.ReLU(),
            nn.Conv2d(24, 36, 5, 2), nn.ReLU(),
            nn.Conv2d(36, 48, 5, 2), nn.ReLU(),
            nn.Conv2d(48, 64, 3), nn.ReLU(),
            nn.Conv2d(64, 64, 3), nn.ReLU(),
            nn.Flatten(),
            nn.LazyLinear(100), nn.ReLU(),
            nn.Linear(100, 50), nn.ReLU(),
            nn.Linear(50, 10), nn.ReLU(),
            nn.Linear(10, 1), nn.Tanh(),
        )

    def forward(self, x):
        return self.model(x)

model = MyModel().to(device)
print(model)

MyModel(
  (model): Sequential(
    (0): Conv2d(3, 12, kernel_size=(5, 5), stride=(2, 2))
    (1): ReLU()
    (2): Conv2d(12, 24, kernel_size=(5, 5), stride=(2, 2))
    (3): ReLU()
    (4): Flatten(start_dim=1, end_dim=-1)
    (5): LazyLinear(in_features=0, out_features=100, bias=True)
    (6): ReLU()
    (7): Linear(in_features=100, out_features=50, bias=True)
    (8): ReLU()
    (9): Linear(in_features=50, out_features=1, bias=True)
    (10): Tanh()
  )
)


In [ ]:
# add optimizer, do a forward pass first so optimizer no params in
# lazy linear layer
model(torch.zeros(1, 3, 1080, 1920, device=device))  # H, W = your input image size
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)